# 05 - Fit the Forecasting Model

**Model choice: multiple linear regression.** Out of the "interpretable regression or tree-based model" options, we're starting with linear regression, fit by ordinary least squares (OLS). Reasons it's the right first model for this project:

- **Fully interpretable** — every feature gets one coefficient, readable directly as "+/− this many covers," which is exactly the language a manager thinks in (unlike a tree ensemble's feature importances, which rank *influence* but don't say direction or size as plainly).
- **Well-suited to the data size** — 638 training rows and ~15 features is comfortable for OLS; tree-based models (Random Forest, gradient boosting) typically want more rows per feature to show their advantage over a linear model, and with under a thousand rows OLS is still the safer, more interpretable starting point.
- **No new dependencies.** We can fit it with `numpy` alone (which you already have), instead of requiring `scikit-learn` to be installed. If you later want to try a Random Forest for comparison, `sklearn.ensemble.RandomForestRegressor` is the natural next step — same features, `pip install scikit-learn` first — and Step 7's caveats below flag exactly where a tree model would likely help most.

**The math, briefly:** OLS finds one weight per feature such that `predicted covers = intercept + weight1×feature1 + weight2×feature2 + ...` has the smallest possible total squared error against the actual covers in the training data. There's a closed-form formula for the best-fitting weights (no trial-and-error, no training loop) — `numpy.linalg.lstsq` solves it directly.

## Step 1: Load features and prepare the design matrix

Two prep steps before fitting:

1. **One-hot encode `day_of_week`.** A model needs numbers, not text — "Saturday" isn't a number linear regression can multiply. One-hot encoding turns it into 6 binary columns (Tuesday, Wednesday, ... Sunday), each 1 if that's the day and 0 otherwise, with **Monday left out as the baseline**. Every day's coefficient is then read as "vs. a Monday," which is the plainest way to interpret it.
2. **Drop `is_weekend`, `month`, `week_of_year` from the model inputs.** `is_weekend` is fully determined by the day-of-week dummies (redundant, and redundant inputs make coefficients unstable). `month`/`week_of_year` are left out of this first version deliberately — notebook 02 showed the seasonal pattern is a *hump* (rises to a May peak, falls back), and a single numeric month column can only represent a straight line, not a hump. We'll rely on the lag/rolling-average features to carry recent-season information instead, and revisit proper seasonality once a second year of data exists.

In [ ]:
import pandas as pd
import numpy as np

mdf = pd.read_csv("../data/processed/covers_model_features.csv", parse_dates=["date"])
mdf = mdf.sort_values("date").reset_index(drop=True)

dow_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
mdf["day_of_week"] = pd.Categorical(mdf["day_of_week"], categories=dow_order)
dow_dummies = pd.get_dummies(mdf["day_of_week"], prefix="dow", drop_first=True).astype(float)  # Monday = baseline

numeric_feats = [
    "covers_lag_7", "covers_lag_14",
    "covers_rollavg_4wk_weekday", "covers_rollavg_7d_trend",
    "is_holiday", "is_local_event", "is_promo",
    "temp_high_f", "is_rain",
]

X = pd.concat([dow_dummies, mdf[numeric_feats]], axis=1)
y = mdf["covers"]

X.head()

## Step 2: Time-based train/test split

Same rule as notebook 03: no shuffling, cutoff date splits train from test. We use the **same 8-week test window** (2026-06-08 onward) as the baseline notebook, on purpose — using an identical test period is what makes "does the model beat the baseline" a fair comparison rather than two numbers from different data.

In [ ]:
test_start = pd.Timestamp("2026-06-08")
train_mask = mdf["date"] < test_start
test_mask = mdf["date"] >= test_start

X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[test_mask], y[test_mask]

print(f"Train: {train_mask.sum()} days, Test: {test_mask.sum()} days")

## Step 3: Fit the model

Add an intercept column (a constant "1" for every row, so the model can fit a baseline level in addition to each feature's effect), then solve for the coefficients with `numpy.linalg.lstsq`.

In [ ]:
X_train_design = np.column_stack([np.ones(len(X_train)), X_train.values])
X_test_design = np.column_stack([np.ones(len(X_test)), X_test.values])

coefs, _, rank, _ = np.linalg.lstsq(X_train_design, y_train.values, rcond=None)
print(f"Solved for {len(coefs)} coefficients (matrix rank {rank} of {X_train_design.shape[1]} — full rank means no exact redundancy)")

## Step 4: Generate test-period forecasts and score them

Same MAE/MAPE metrics as notebook 03, and the same test period — so this number drops in directly next to the 19.0 MAE / 12.9% MAPE baseline.

In [ ]:
pred_train = X_train_design @ coefs
pred_test = X_test_design @ coefs

def mae(pred, actual):
    return np.mean(np.abs(pred - actual))

def mape(pred, actual):
    return np.mean(np.abs(pred - actual) / actual) * 100

print("Model  — Test MAE:", round(mae(pred_test, y_test.values), 1),
      " Test MAPE:", round(mape(pred_test, y_test.values), 1), "%")
print("Baseline (4-wk rolling avg, from notebook 03) — MAE: 19.0  MAPE: 12.9%")

ss_res = np.sum((y_test.values - pred_test) ** 2)
ss_tot = np.sum((y_test.values - y_test.values.mean()) ** 2)
r2_test = 1 - ss_res / ss_tot
print(f"Test R²: {r2_test:.3f}  (share of test-period variation the model explains)")

**Read this as:** the model beats the baseline on both metrics — a meaningful drop in typical error, in covers a manager would actually feel in staffing. R² of ~0.83 means the model explains about 83% of the swing in test-period covers; the remaining 17% is noise and one-off events no feature captures (more on that in Step 6).

## Step 5: Interpret the coefficients

Each coefficient is "how many covers this feature adds or subtracts, holding everything else in the model constant." Sort them to read like a rate card.

In [ ]:
coef_names = ["intercept"] + list(X.columns)
coef_table = pd.Series(coefs, index=coef_names).round(1).sort_values(ascending=False)
coef_table

**How a manager reads this table:**

- The `dow_*` rows are the biggest movers by far — Saturday adds about 121 covers vs. a Monday, Friday about 95 — matching notebook 02's finding almost exactly. This is the model confirming the weekday pattern is real and dominant.
- `is_holiday` and `is_local_event` both add a clear, positive lift — worth specifically staffing up for when the calendar flags one.
- `covers_lag_14`, `covers_lag_7`, and `temp_high_f` all have small positive coefficients — real, but minor nudges rather than headline drivers, matching their weak correlations from notebook 02.

**Two coefficients that look off, and why:** `is_rain` and `covers_rollavg_4wk_weekday` may come out negative or oddly sized, even though notebook 02/03 suggested they help. This is **multicollinearity** — several features here move together (a Saturday's rolling weekday average is *always* high, so it overlaps heavily with the `dow_Saturday` dummy; rain and temperature also move together with the season). When two inputs carry overlapping information, OLS can split credit between them in a way that makes one look small or backwards, even though the *combined* effect is still right — which is exactly why the overall test MAE/MAPE above are still solid. It's a real limitation of linear regression with correlated inputs, and it's one place a tree-based model (which doesn't need to "split credit" the same way) would likely give cleaner individual feature reads if you want to compare later.

## Step 6: Where the model does well or poorly

Look at the largest test-period misses, the same way notebook 03 did for the baselines.

In [ ]:
test_df = mdf[test_mask].copy()
test_df["predicted"] = np.round(pred_test, 1)
test_df["abs_err"] = (test_df["predicted"] - test_df["covers"]).abs()

worst = test_df.sort_values("abs_err", ascending=False).head(8)
worst[["date", "day_of_week", "covers", "predicted", "abs_err",
       "is_holiday", "is_local_event", "is_promo", "is_rain", "temp_high_f"]]

**Operational read of the misses:** several of the worst days are Saturdays where a 200+ cover night landed a bit above or below what the model expected — the model gets the *scale* right (it knows Saturdays run big) but not the exact number on an unusually hot, rainy, or otherwise atypical Saturday. One miss (a Monday at 41 covers vs. a much higher prediction) traces back to a midday closure for a health inspection — a real-world disruption with **no feature in the dataset at all**, since nobody logs "closed for an inspection" as a planning input. That's a useful, honest limit to communicate to a manager: the forecast is a strong starting point for a normal day, but large one-off disruptions still need human judgment layered on top.

## Step 7: From forecast to a staffing/inventory decision

The model outputs a number of covers; a manager needs to translate that into people and product. Two concrete examples using this restaurant's numbers:

- **Staffing:** if the floor runs comfortably at roughly 1 server per 20 covers, a forecast of 220 covers suggests staffing about 11 servers, versus roughly 5 for a 100-cover Monday — the model turns "it'll probably be busy Saturday" into a specific number to schedule against.
- **Inventory/prep:** if a menu item historically sells at a rate of, say, 0.4 orders per cover, a 220-cover forecast implies prepping for roughly 88 orders of that item, scaled the same way for any dish with a known per-cover rate.

Both examples work off the same forecast number — the model doesn't need to know about staffing ratios or prep pars itself, it just needs to be a trustworthy covers estimate that a manager then runs through whatever ratios their operation already uses.

## Recap

- Fit **multiple linear regression** (OLS via `numpy.linalg.lstsq`) — interpretable, no new dependencies, well-matched to a 638-row training set.
- Used the **same test window as the baseline** for a fair comparison: model MAE/MAPE beat the 19.0 / 12.9% baseline from notebook 03.
- **Day of week dominates** the coefficients, consistent with every prior notebook; holidays and local events add a clear, positive, actionable lift.
- **Multicollinearity** (rolling weekday average vs. day-of-week dummies; rain vs. temperature) makes a couple of individual coefficients look counterintuitive even though overall accuracy is solid — flagged as a place a tree-based model could improve on later.
- The model still misses **unflagged, one-off disruptions** (like a surprise closure) — a reminder that this is a decision-support tool, not a replacement for a manager's judgment on unusual days.
- Translated a forecast into concrete staffing/inventory math a manager could actually use.

**Next up:** package this into a simple weekly workflow — a notebook or spreadsheet a manager updates with upcoming dates, reservations, events, and a weather forecast, to get next week's covers forecast without touching any of the modeling code directly.